In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install -q google-genai

In [ ]:
import re

import pandas as pd
import google.generativeai as genai
from google.genai.types import HarmCategory, HarmBlockThreshold
import time

from math import ceil


In [ ]:
SELECTED_MODEL = 'gemini-2.5-pro-preview-03-25'
max_output_tokens =  65536

batch_size = 100

In [ ]:
sample_counts = {
    'False,NEG,egypt': 1000,
    'False,NEG,gulf': 1000,
    'False,NEG,levant': 1000,
    'False,NEG,magreb': 1000,
    'False,NEG,msa': 1000,
    'False,NEU,egypt': 1000,
    'False,NEU,gulf': 1000,
    'False,NEU,levant': 1000,
    'False,NEU,magreb': 1000,
    'False,NEU,msa': 1000,
    'False,POS,egypt': 1000,
    'False,POS,gulf': 1000,
    'False,POS,levant': 1000,
    'False,POS,magreb': 1000,
    'False,POS,msa': 1000,
    'True,NEG,egypt': 1500,
    'True,NEG,gulf': 1500,
    'True,NEG,levant': 1500,
    'True,NEG,magreb': 1500,
    'True,NEG,msa': 1500,
    'True,NEU,egypt': 1500,
    'True,NEU,gulf': 1500,
    'True,NEU,levant': 1500,
    'True,NEU,magreb': 1500,
    'True,NEU,msa': 1500
}

In [ ]:
%%capture
!pip install camel-tools
!pip install -q urlextract
!pip install PyArabic

from camel_tools.utils.normalize import normalize_unicode
from camel_tools.utils.normalize import normalize_alef_maksura_ar
from camel_tools.utils.normalize import normalize_alef_ar
from camel_tools.utils.normalize import normalize_teh_marbuta_ar
from camel_tools.utils.dediac import dediac_ar

from urlextract import URLExtract

from pyarabic.araby import strip_tatweel

In [ ]:
arabic_to_latin_numbers = { '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9' }

extractor = URLExtract()

def clean_text(text):
  text = normalize_unicode(text) # ﷺ -> صلى الله عليه وسلم
  text = normalize_alef_ar(text) # Normalize alef variants to 'ا'
  text = normalize_alef_maksura_ar(text) # Normalize alef maksura 'ى' to yeh 'ي'
  text = normalize_teh_marbuta_ar(text) # Normalize teh marbuta 'ة' to heh 'ه'
  text = dediac_ar(text) # Dediacritization
  text = text.translate(str.maketrans(arabic_to_latin_numbers))

  urls = extractor.find_urls(text)
  for url in urls:
      text = text.replace(url, '')

  text = strip_tatweel(text)    # e.g. الســـــــــــــــــــعودية -> السعودية

  text = re.sub(r'(.)\1{2,}', r'\1\1', text)  # Replace 3+ consecutive chars with just 2

  # text = re.sub(r'[^ء-يa-zA-Z0-9\s\(\)\[\]\{\}]+', ' ', text) # Remove all characters except Arabic letters, English letters, digits, # whitespace, and specific punctuation marks (parentheses, brackets, braces).
  text = re.sub(r'[^ء-ي\s]+', ' ', text)
  text = re.sub(r'\s+', ' ', text).strip() # Collapse multiple spaces into a single space and trim leading/trailing spaces.

  return text

In [ ]:
def create_model():
    with open('/content/drive/MyDrive/Project/secrets/Gemini-API-Key.txt', 'r') as f:
        GEMINI_API_KEY = f.read().strip()

    safety_settings_options = {
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    }

    genai.configure(api_key=GEMINI_API_KEY)

    generation_config = {
        "temperature": 1,  # Controls randomness (0 = deterministic, 1 = highly random)
        "top_p": 0.95,     # Higher values make the model more creative
        "max_output_tokens": max_output_tokens,
        "response_mime_type": "text/plain",
    }

    model = genai.GenerativeModel(
        model_name=SELECTED_MODEL,
        generation_config=generation_config,
        safety_settings=safety_settings_options,
    )
    print(model)
    return model

model =     create_model()

genai.GenerativeModel(
    model_name='models/gemini-2.5-pro-preview-03-25',
    generation_config={'temperature': 1, 'top_p': 0.95, 'max_output_tokens': 65536, 'response_mime_type': 'text/plain'},
    safety_settings={<HarmCategory.HARM_CATEGORY_HARASSMENT: 7>: <HarmBlockThreshold.BLOCK_NONE: 4>, <HarmCategory.HARM_CATEGORY_HATE_SPEECH: 8>: <HarmBlockThreshold.BLOCK_NONE: 4>, <HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 9>: <HarmBlockThreshold.BLOCK_NONE: 4>, <HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 10>: <HarmBlockThreshold.BLOCK_NONE: 4>},
    tools=None,
    system_instruction=None,
    cached_content=None
)


In [ ]:
def create_prompt(sarcasm, sentiment, dialect):
  if sarcasm:
    return f'''I will give you Arabic sarcastic tweets analysis and answers to the below questions:
    - The topic of the tweet.
    - The context of the tweet.
    - The dialect of the tweet.
    Dialect can be one of the following:
    * msa: Modern Standard Arabic (الفصحى) used in formal writing and media,
    * egypt: Egyptian dialect (العامية المصرية) used in Egypt and Sudan,
    * levant: Levantine dialect (اللهجة الشامية) used in Palestine, Jordan, Syria and Lebanon,
    * gulf: Gulf dialect (اللهجة الخليجية) used in Saudi Arabia, UAE, Qatar, Bahrain, Yemen, Oman, Iraq and Kuwait,
    * magreb: Maghrebi dialect (اللهجة المغاربية) used in North African Arab countries including Algeria, Libya, Tunisia and Morocco
    - Year was this tweet written based on the topic/context of it.
    - Country was that tweet written.
    - Why tweet is sarcastic.
    - Type of sarcasm used?
    - Impolite words.
    - Hateful words.
    - Contradictory statements.
    - Hidden meanings.
    - Double entendre.
    - Metaphors to compare things to other things.
    - Long or a short tweet. The best lengh of the tweet.
    - Number of arabic words used.
    - Number of non-Arabic words used.
    - Number of hashtags or urls or special characters used.
    - Analysis of the personality and the circumstances of the person who wrote the tweet.
    - Motives led this person to say this tweet.
    - The relationship between dialect and country and year and context of tweet and sarcasm used and personality and the circumstances.

    Your task is to write the original tweet that has this analysis but using {sentiment} sentiment and {dialect} dialect and 25% new words. Response must have the tweet and nothing else.
    Never use ambiguous Arabic words such as فلان او علان, instead, create real names and words that suit the tweet context.
    Never mask bad words or hateful words with characters like ** or anything else.
    Keep the same exact order of the tweets as the passed analysis. Each tweet must be in exactelly 1 line that start and end with tweet text and no other string. Do not include in response any explanation or further strings.
    Number of response lines must = number of passed analysis.
    Each tweet must not start or end with a new line \n
    '''

  return f'''I will give you Arabic tweets analysis and answers to the below questions:
  - The topic of the tweet.
  - The context of the tweet.
  - The dialect of the tweet.
  Dialect can be one of the following:
  * msa: Modern Standard Arabic (الفصحى) used in formal writing and media,
  * egypt: Egyptian dialect (العامية المصرية) used in Egypt and Sudan,
  * levant: Levantine dialect (اللهجة الشامية) used in Palestine, Jordan, Syria and Lebanon,
  * gulf: Gulf dialect (اللهجة الخليجية) used in Saudi Arabia, UAE, Qatar, Bahrain, Yemen, Oman, Iraq and Kuwait,
  * magreb: Maghrebi dialect (اللهجة المغاربية) used in North African Arab countries including Algeria, Libya, Tunisia and Morocco
  - Year was this tweet written based on the topic/context of it.
  - Country was that tweet written.
  - Impolite words.
  - Hateful words.
  - Contradictory statements.
  - Hidden meanings.
  - Double entendre.
  - Metaphors to compare things to other things.
  - Long or a short tweet. The best lengh of the tweet.
  - Number of arabic words used.
  - Number of non-Arabic words used.
  - Number of hashtags or urls or special characters used.
  - Analysis of the personality and the circumstances of the person who wrote the tweet.
  - Motives led this person to say this tweet.
  - The relationship between dialect and country and year and context of tweet and personality and the circumstances.

  Your task is to write the original tweet that has this analysis but using {sentiment} sentiment and {dialect} dialect and 25% new words. Response must have the tweet and nothing else.
  Never use ambiguous Arabic words such as فلان او علان, instead, create real names and words that suit the tweet context.
  Never mask bad words or hateful words with characters like ** or anything else.
  Keep the same exact order of the tweets as the passed analysis. Each tweet must be in exactelly 1 line that start and end with tweet text and no other string. Do not include in response any explanation or further strings.
  Number of response lines must = number of passed analysis.
  Each tweet must not start or end with a new line \n

  '''

In [ ]:
def generate_tweet_batch(prompt, analysis, chat_gen):
    try:
        full_prompt = prompt + '\n' + '\n'.join(analysis)
        response = chat_gen.send_message(full_prompt).text

        # Clean up the response
        tweets = re.sub(r'\n\s*\n', '\n', response)  # Remove empty lines
        tweets = tweets.split('\n')

        # Filter out empty tweets and clean them
        tweets = [tweet.strip() for tweet in tweets if tweet.strip()]

        # Ensure we return exactly the same number of tweets as analysis samples
        expected_count = len(analysis)
        if len(tweets) != expected_count:
            print(f"WARNING: Expected {expected_count} tweets, got {len(tweets)}")

            if len(tweets) > expected_count:
                tweets = tweets[:expected_count]
                print(f"Truncated to {expected_count} tweets")
            elif len(tweets) < expected_count:
                # Pad with the last tweet or a placeholder
                while len(tweets) < expected_count:
                    if tweets:
                        tweets.append(tweets[-1])  # Repeat last tweet
                    else:
                        tweets.append("Generated tweet placeholder")
                print(f"Padded to {expected_count} tweets")

        return tweets

    except Exception as e:
        print(f"Error in tweet generation: {str(e)}")
        # Return placeholder tweets to maintain count
        return [f"Error generating tweet {i+1}" for i in range(len(analysis))]

In [ ]:
def generate_tweet_batch(prompt, analysis, chat_gen):
    return analysis


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/analysis.csv')
grouped = df.groupby('combined_label')
all_generated_data = []

for combined_label, group_data in grouped:
    target_count = sample_counts.get(combined_label, 0)
    current_count = len(group_data)

    if current_count < target_count:
        needed_samples = target_count - current_count

        print(f"\n{'='*60}")
        print(f"Current count: {current_count}, Target count: {target_count}, Needed samples: {needed_samples}")
        print(f"Processing group: {combined_label}")
        print(f"{'='*60}")

        sample_row = group_data.iloc[0]
        sarcasm = sample_row['sarcasm']
        sentiment = sample_row['sentiment']
        dialect = sample_row['dialect']

        # Load analysis data from entire dataset based on sarcasm value
        if sarcasm:
            sarcastic_data = df[df['sarcasm'] == True]
            analysis_data = sarcastic_data['analysis'].tolist()
        else:
            non_sarcastic_data = df[df['sarcasm'] == False]
            analysis_data = non_sarcastic_data['analysis'].tolist()

        print(f"Loaded {len(analysis_data)} analysis samples for sarcasm={sarcasm}")

        # Check if we have enough analysis data
        if len(analysis_data) == 0:
            print(f"ERROR: No analysis data found for sarcasm={sarcasm}")
            continue

        generated_count = 0
        num_batches = ceil(needed_samples / batch_size)

        for batch_num in range(num_batches):
            time.sleep(2)  # Rate limiting

            # Calculate samples for this batch
            remaining_samples = needed_samples - generated_count
            samples_in_batch = min(batch_size, remaining_samples)

            print(f"\nProcessing batch {batch_num + 1}/{num_batches} with {samples_in_batch} samples")

            # Get batch of analysis data (cycle through if needed)
            batch_analysis = []
            for i in range(samples_in_batch):
                idx = (batch_num * batch_size + i) % len(analysis_data)
                batch_analysis.append(analysis_data[idx])

            print(f"Selected {len(batch_analysis)} analysis samples for this batch")

            # Create prompt
            base_prompt = create_prompt(sarcasm, sentiment, dialect)

            try:
                # Generate tweets for this batch
                chat_gen = model.start_chat(history=[])
                generated_tweets = generate_tweet_batch(base_prompt, batch_analysis, chat_gen)

                print(f"Successfully generated {len(generated_tweets)} tweets")

                # Verify counts match
                if len(generated_tweets) != len(batch_analysis):
                    print(f"ERROR: Mismatch - {len(generated_tweets)} tweets vs {len(batch_analysis)} analysis")

                # Create DataFrame for this batch
                batch_data = []
                for i in range(len(batch_analysis)):
                    tweet_text = generated_tweets[i] if i < len(generated_tweets) else "Missing tweet"

                    batch_data.append({
                        'sarcasm': sarcasm,
                        'sentiment': sentiment,
                        'dialect': dialect,
                        'combined_label': combined_label,
                        'text': tweet_text.strip() if tweet_text else "Empty tweet"
                    })

                # Add to all generated data
                all_generated_data.extend(batch_data)
                generated_count += len(batch_data)

                print(f"Added {len(batch_data)} records. Total generated: {generated_count}/{needed_samples}")

            except Exception as e:
                print(f"Error in batch {batch_num + 1}: {str(e)}")
                # Create placeholder data for failed batch
                batch_data = []
                for i in range(samples_in_batch):
                    batch_data.append({
                        'sarcasm': sarcasm,
                        'sentiment': sentiment,
                        'dialect': dialect,
                        'combined_label': combined_label,
                        'analysis': batch_analysis[i] if i < len(batch_analysis) else "No analysis",
                        'text': f"Error generating tweet for batch {batch_num + 1}, sample {i + 1}"
                    })

                all_generated_data.extend(batch_data)
                generated_count += len(batch_data)
                print(f"Added {len(batch_data)} error placeholders")

            # Break if we've generated enough samples
            if generated_count >= needed_samples:
                break

print(f"\n{'='*60}")
print(f"Generation complete. Total records: {len(all_generated_data)}")

# Save results
if all_generated_data:
    generated_df = pd.DataFrame(all_generated_data)
    generated_df['text'] = generated_df['text'].apply(clean_text)

    # Check for empty text columns
    empty_text_count = generated_df['text'].str.strip().eq('').sum()
    if empty_text_count > 0:
        print(f"WARNING: {empty_text_count} records have empty text")

    # Save to CSV
    output_path = '/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/generated.csv'
    generated_df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Saved {len(generated_df)} records to {output_path}")

    # Print summary
    print("\nGeneration Summary:")
    print(generated_df.groupby('combined_label').size())

else:
    print("No data generated!")



Current count: 486, Target count: 1000, Needed samples: 514
Processing group: False,NEG,egypt
Loaded 8623 analysis samples for sarcasm=False

Processing batch 1/6 with 100 samples
Selected 100 analysis samples for this batch
Successfully generated 100 tweets
Added 100 records. Total generated: 100/514

Processing batch 2/6 with 100 samples
Selected 100 analysis samples for this batch
Successfully generated 100 tweets
Added 100 records. Total generated: 200/514

Processing batch 3/6 with 100 samples
Selected 100 analysis samples for this batch
Successfully generated 100 tweets
Added 100 records. Total generated: 300/514

Processing batch 4/6 with 100 samples
Selected 100 analysis samples for this batch
Successfully generated 100 tweets
Added 100 records. Total generated: 400/514

Processing batch 5/6 with 100 samples
Selected 100 analysis samples for this batch
Successfully generated 100 tweets
Added 100 records. Total generated: 500/514

Processing batch 6/6 with 14 samples
Selected 1

In [ ]:
import time
def shutdown_sys():
  print('Will Shutdown')
  time.sleep(30)
  from google.colab import runtime
  runtime.unassign()

In [ ]:
shutdown_sys()

Will Shutdown
